# Spambase Spam Classification

4,601 emails, each described by 57 word-frequency, character-frequency and capital-run-length features. Predict spam (1) versus non-spam (0) — the classic email-filtering benchmark, without reading any message text.

In [1]:
import io, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def download(url):
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=180) as r:
        return r.read()
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

# UCI Spambase (id 94)
df = pd.read_csv(io.BytesIO(download('https://archive.ics.uci.edu/static/public/94/data.csv')))
print(df.shape)
print('Emails per class (1 = spam, 0 = not spam):')
print(df['Class'].value_counts())
df.head(3)


(4601, 58)
Emails per class (1 = spam, 0 = not spam):
Class
0    2788
1    1813
Name: count, dtype: int64


,word_freq_make,word_freq_address,word_freq_all,word_freq_3d,word_freq_our,word_freq_over,word_freq_remove,word_freq_internet,word_freq_order,word_freq_mail,...,char_freq_;,char_freq_(,char_freq_[,char_freq_!,char_freq_$,char_freq_#,capital_run_length_average,capital_run_length_longest,capital_run_length_total,Class
0,0.00,0.64,0.64,0.0,0.32,0.00,0.00,0.00,0.00,0.00,...,0.00,0.000,0.0,0.778,0.000,0.000,3.756,61,278,1
1,0.21,0.28,0.50,0.0,0.14,0.28,0.21,0.07,0.00,0.94,...,0.00,0.132,0.0,0.372,0.180,0.048,5.114,101,1028,1
2,0.06,0.00,0.71,0.0,1.23,0.19,0.19,0.12,0.64,0.25,...,0.01,0.143,0.0,0.276,0.184,0.010,9.821,485,2259,1


In [2]:

feats = [c for c in df.columns if c != 'Class']
X = df[feats]; y = df['Class']
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1).fit(Xtr, ytr)
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xtr, ytr)
acc_rf = accuracy_score(yte, rf.predict(Xte)); acc_lr = accuracy_score(yte, lr.predict(Xte))
print(f'Random Forest accuracy: {acc_rf:.4f}')
print(f'Logistic Regression accuracy: {acc_lr:.4f}')
print(classification_report(yte, rf.predict(Xte)))


Random Forest accuracy: 0.9531
Logistic Regression accuracy: 0.9279
              precision    recall  f1-score   support

           0       0.95      0.98      0.96       697
           1       0.96      0.92      0.94       454

    accuracy                           0.95      1151
   macro avg       0.96      0.95      0.95      1151
weighted avg       0.95      0.95      0.95      1151



In [3]:
imp = pd.Series(rf.feature_importances_, index=feats).sort_values(ascending=False)
print(imp.head(20))
imp.head(20).plot(kind='barh', title='Top 20 spam-indicating features')
plt.tight_layout()
plt.show()
winner = rf if acc_rf >= acc_lr else lr
winner_name = 'Random Forest' if acc_rf >= acc_lr else 'Logistic Regression'
print(f'Winner: {winner_name}')
cm = confusion_matrix(yte, winner.predict(Xte), labels=sorted(y.unique()))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=sorted(y.unique()))
disp.plot(xticks_rotation=45)
plt.title(f'Confusion matrix - {winner_name}')
plt.tight_layout()
plt.savefig('confusion_matrix.png')
plt.show()


char_freq_!                   0.115637
char_freq_$                   0.106525
word_freq_remove              0.073909
capital_run_length_average    0.060269
word_freq_your                0.057299
word_freq_free                0.057296
capital_run_length_longest    0.055048
capital_run_length_total      0.054136
word_freq_hp                  0.045352
word_freq_money               0.034371
word_freq_our                 0.030862
word_freq_you                 0.030816
word_freq_000                 0.027040
word_freq_george              0.018738
word_freq_hpl                 0.016962
word_freq_edu                 0.016577
word_freq_internet            0.013053
char_freq_(                   0.012976
word_freq_1999                0.012227
word_freq_will                0.011957
dtype: float64


Winner: Random Forest
